# Bibliotecas

In [1]:
import pandas as pd
import yahooquery as yq

# Leitura e tratamento dos dados

DATA COM: É dada que se pode comprar para receber o pagamento referido na linha

DY: É o valor do dividendo sobre o valor da ação

In [10]:
planilha = pd.read_csv("Dados Capturados/(Cru) FIIs rendimentos por data (2021-2025).csv")
planilha

,ATIVO,VALOR,DATA COM,DATA PAGAMENTO,TIPO,DY
0,ARRI11,"R$ 0,08018867",30/12/2020,08/01/2021,Rendimento,"0,09%"
1,BRCO11,"R$ 0,55000000",30/12/2020,08/01/2021,Rendimento,"0,48%"
2,HSAF11,"R$ 0,70000000",30/12/2020,08/01/2021,Rendimento,"0,74%"
3,HSML11,"R$ 0,50000000",30/12/2020,08/01/2021,Rendimento,"0,53%"
4,LVBI11,"R$ 0,47000000",30/12/2020,08/01/2021,Rendimento,"0,39%"
...,...,...,...,...,...,...
5606,VGIR11,"R$ 0,13000000",10/09/2025,17/09/2025,Rendimento,"1,34%"
5607,AJFI11,"R$ 0,06000000",05/09/2025,19/09/2025,Rendimento,"0,78%"
5608,MANA11,"R$ 0,11000000",29/08/2025,19/09/2025,Rendimento,"1,27%"
5609,SNEL11,"R$ 0,10000000",15/09/2025,25/09/2025,Rendimento,"0,00%"


In [11]:
planilha.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5611 entries, 0 to 5610
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   ATIVO           5611 non-null   object
 1   VALOR           5611 non-null   object
 2   DATA COM        5611 non-null   object
 3   DATA PAGAMENTO  5611 non-null   object
 4   TIPO            5611 non-null   object
 5   DY              5611 non-null   object
dtypes: object(6)
memory usage: 263.1+ KB


In [12]:
planilha["VALOR"] = planilha["VALOR"].apply(lambda x: x.replace("R$ ", ""))
# planilha["VALOR"] = planilha["VALOR"].apply(lambda x: float(x.split(',')[0] + '.' + x.split(',')[1][:2]))
planilha["DATA COM"] =  planilha["DATA COM"].apply(lambda x: pd.to_datetime(x.replace('/', '-'), format='%d-%m-%Y'))
planilha["DATA PAGAMENTO"] =  planilha["DATA PAGAMENTO"].apply(lambda x: pd.to_datetime(x.replace('/', '-'), format='%d-%m-%Y'))
planilha["DY"] =  planilha["DY"].apply(lambda x: x.replace('%', ''))
planilha

,ATIVO,VALOR,DATA COM,DATA PAGAMENTO,TIPO,DY
0,ARRI11,"0,08018867",2020-12-30,2021-01-08,Rendimento,"0,09"
1,BRCO11,"0,55000000",2020-12-30,2021-01-08,Rendimento,"0,48"
2,HSAF11,"0,70000000",2020-12-30,2021-01-08,Rendimento,"0,74"
3,HSML11,"0,50000000",2020-12-30,2021-01-08,Rendimento,"0,53"
4,LVBI11,"0,47000000",2020-12-30,2021-01-08,Rendimento,"0,39"
...,...,...,...,...,...,...
5606,VGIR11,"0,13000000",2025-09-10,2025-09-17,Rendimento,"1,34"
5607,AJFI11,"0,06000000",2025-09-05,2025-09-19,Rendimento,"0,78"
5608,MANA11,"0,11000000",2025-08-29,2025-09-19,Rendimento,"1,27"
5609,SNEL11,"0,10000000",2025-09-15,2025-09-25,Rendimento,"0,00"


In [13]:
planilha.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5611 entries, 0 to 5610
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   ATIVO           5611 non-null   object        
 1   VALOR           5611 non-null   object        
 2   DATA COM        5611 non-null   datetime64[ns]
 3   DATA PAGAMENTO  5611 non-null   datetime64[ns]
 4   TIPO            5611 non-null   object        
 5   DY              5611 non-null   object        
dtypes: datetime64[ns](2), object(4)
memory usage: 263.1+ KB


In [ ]:
planilha.to_csv("Dados Modelados/FIIs rendimentos por data (2021-2025).csv", index=False)

# Criando historico preços

In [14]:
ativos = pd.unique(planilha.loc[:, "ATIVO"]).tolist()

In [15]:
lista_tickers = []
for ativo in ativos:
    lista_tickers.append((ativo,yq.Ticker(ativo+'.SA')))

In [8]:
idx = 0
lista_precos_historicos = []
for nome, ticker in lista_tickers:
    lista_precos_historicos.append(ticker.history(period='5y',interval='1mo'))
    lista_precos_historicos[idx].assign(ativo=nome)
    idx += 1

In [9]:
df_historico_ativos = pd.concat(lista_precos_historicos)

C:\Users\luis.nascimento\AppData\Local\Temp\ipykernel_15544\3605042950.py:1: RuntimeWarning: can't compare datetime.datetime to datetime.date, sort order is undefined for incomparable objects.
  df_historico_ativos = pd.concat(lista_precos_historicos)


In [10]:
df_historico_ativos['Ativo'] = df_historico_ativos.index.get_level_values(0)
df_historico_ativos['Data'] = df_historico_ativos.index.get_level_values(1)

In [11]:
for idx, line in df_historico_ativos.iterrows():
    df_historico_ativos.loc[idx,'Ativo'] = df_historico_ativos.loc[idx,'Ativo'].split('.')[0]

In [12]:
df_historico_ativos['Data'] = pd.to_datetime(df_historico_ativos['Data'], format='%Y-%m-%d')

C:\Users\luis.nascimento\AppData\Local\Temp\ipykernel_15544\4055562379.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_historico_ativos['Data'] = pd.to_datetime(df_historico_ativos['Data'], format='%Y-%m-%d')


In [13]:
df_historico_ativos['Data'] = df_historico_ativos['Data'].apply(lambda x: x.strftime('%d-%m-%Y'))

In [14]:
df_historico_ativos.head(10)

open       high        low      close    volume  \
symbol    date                                                               
ARRI11.SA 2020-10-01   9.052735   9.052735   8.169034   8.807318   53153.0   
          2020-11-01   8.807318   9.321903   8.312524   9.302111   86916.0   
          2020-12-01   9.222944   9.400080   8.609400   9.183360  137256.0   
          2021-01-01   9.183360   9.598987   8.808307   9.401069  129692.0   
          2021-02-01   9.303100   9.994821   9.303100   9.993832  158645.0   
          2021-03-01   9.871123   9.876071   9.336746   9.876071  315032.0   
          2021-04-01   9.885967  11.201127   9.747425  11.201127  266264.0   
          2021-05-01  10.885449  11.324825   9.922581   9.961175  405688.0   
          2021-06-01  10.006696  10.390656   9.845394  10.141280  478908.0   
          2021-07-01  10.191749  11.281283  10.045290  11.266440  356209.0   

                      adjclose  dividends  splits   Ativo        Data  
symbol    date                                                         
ARRI11.SA 2020-10-01  5.453424        0.0     0.0  ARRI11  01-10-2020  
          2020-11-01  5.759795        0.0     0.0  ARRI11  01-11-2020  
          2020-12-01  5.686265        0.0     0.0  ARRI11  01-12-2020  
          2021-01-01  5.821070        0.0     0.0  ARRI11  01-01-2021  
          2021-02-01  6.188105        0.0     0.0  ARRI11  01-02-2021  
          2021-03-01  6.115188        0.0     0.0  ARRI11  01-03-2021  
          2021-04-01  6.935653        0.0     0.0  ARRI11  01-04-2021  
          2021-05-01  6.167882        0.0     0.0  ARRI11  01-05-2021  
          2021-06-01  6.279401        0.0     0.0  ARRI11  01-06-2021  
          2021-07-01  6.976094        0.0     0.0  ARRI11  01-07-2021

In [15]:
df_historico_ativos.to_csv('Dados Modelados/historico_Fii.csv', index=False)

# Criando setor, insdustria e recomendação

In [16]:
idx = 0
df_fiis = pd.DataFrame()
for nome, ticker in lista_tickers:
    df_fiis.loc[idx, 'ATIVO'] = nome
    df_fiis.loc[idx, 'SETOR'] = ticker.asset_profile[nome+'.SA']['industry']
    df_fiis.loc[idx, 'INDUSTRIA'] = ticker.asset_profile[nome+'.SA']['sector']
    idx += 1

In [17]:
df_fiis

,ATIVO,SETOR,INDUSTRIA
0,ARRI11,REIT - Residential,Real Estate
1,BRCO11,REIT - Industrial,Real Estate
2,HSAF11,REIT - Diversified,Real Estate
3,HSML11,Asset Management,Financial Services
4,LVBI11,Real Estate Services,Real Estate
...,...,...,...
110,BBIG11,REIT - Retail,Real Estate
111,AZPL11,REIT - Industrial,Real Estate
112,JSCR11,REIT - Diversified,Real Estate
113,TOPP11,REIT - Diversified,Real Estate


In [18]:
df_fiis.to_csv('Dados Modelados/FIIs.csv', index=False)